# 02 — Inference (5 arms × N models × 200 questions)

Run all inference combos. Use T4 GPU + parallel workers (1 process per combo).

**Key advantages over local laptop**:
- T4 16GB VRAM hosts 4-8 BGE-M3 instances concurrent (vs 1 on 4GB Laptop GPU)
- No Windows paging file segfault on parallel torch load
- Faster CPU for SWI-Prolog subprocess

**Limits**:
- OpenAI RPM/TPM per tier — Tier 4+ recommended for 4-8 concurrent
- Colab session 12-24h — split into batches nếu cần

Outputs go to `data/eval/results/{arm}/A*.json` and `data/eval/multimodel/results/{combo}/A*.json`.

## 2.1 Re-setup environment

(Skip nếu chạy ngay sau NB1 cùng session.)

In [ ]:
from google.colab import drive, userdata
import os, sys
drive.mount('/content/drive')
%cd /content/legal-graph-kb
sys.path.insert(0, '/content/legal-graph-kb')

for k in ['OPENAI_API_KEY', 'NEO4J_URI', 'NEO4J_USER', 'NEO4J_PASSWORD']:
    os.environ[k] = userdata.get(k) or ''
os.environ['NEO4J_DATABASE'] = 'neo4j'
os.environ['EMBED_DEVICE'] = 'cuda'
os.environ['PYTHONIOENCODING'] = 'utf-8'
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv

## 2.2 Configure run scope

Pick which combos to run. Default = full experiment matrix.

In [ ]:
# R1 (5-arm với gpt-4o-mini)
R1_ARMS = ['graphrag', 'llm_only', 'elite_no_retrieval', 'elite_ontology', 'elite_graphrag']
# R2 (multi-model, 2 elite arms)
R2_MODELS = ['gpt-4.1', 'gpt-4o', 'gpt-5-mini']
R2_ARMS = ['elite_no_retrieval', 'elite_graphrag']

N_QUESTIONS = 200  # max 200, lower for pilot

# Parallelism config
MAX_WORKERS_R1 = 5     # 5 arms parallel (T4 16GB headroom)
MAX_WORKERS_R2 = 4     # 6 combos in 2 batches of 3-4

print(f'R1: {len(R1_ARMS)} arms × {N_QUESTIONS} questions = {len(R1_ARMS)*N_QUESTIONS} inferences')
print(f'R2: {len(R2_MODELS)*len(R2_ARMS)} combos × {N_QUESTIONS} = {len(R2_MODELS)*len(R2_ARMS)*N_QUESTIONS} inferences')

## 2.3 Parallel runner — helper

Spawn 1 subprocess per (arm, model). Each runs `run_inference.py` / `run_multimodel_inference.py` independently.
Idempotent (skip records đã có).

In [ ]:
import subprocess, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

os.makedirs('/content/legal-graph-kb/logs', exist_ok=True)

def run_combo(cmd, log_name):
    """Run shell cmd, save log, return (cmd, exit_code, elapsed_s)."""
    log_path = f'/content/legal-graph-kb/logs/{log_name}.log'
    t0 = time.time()
    with open(log_path, 'w', encoding='utf-8') as f:
        proc = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)
    return log_name, proc.returncode, time.time() - t0

def parallel_run(cmds_with_names, max_workers):
    """Run list of (cmd, name) tuples parallel, print results as they finish."""
    t_start = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(run_combo, cmd, name): name for cmd, name in cmds_with_names}
        for fut in as_completed(futs):
            name, code, dur = fut.result()
            status = '✓' if code == 0 else f'✗ exit={code}'
            print(f'[{time.strftime("%H:%M:%S")}] {status} {name} ({dur:.0f}s)')
    print(f'\nTotal wall time: {(time.time()-t_start)/60:.1f} min')

## 2.4 R1 — 5 arms parallel

All 5 arms run concurrently. Each uses gpt-4o-mini.

In [ ]:
r1_cmds = [
    (f'python -m experiments.run_inference --arm {arm} --n {N_QUESTIONS}',
     f'r1_{arm}')
    for arm in R1_ARMS
]
parallel_run(r1_cmds, max_workers=MAX_WORKERS_R1)

## 2.5 R2 — 6 combos parallel

All 6 (arm × model) combos concurrent. Stagger model loads (gpt-5-mini reasoning slow).

In [ ]:
r2_cmds = []
for arm in R2_ARMS:
    for model in R2_MODELS:
        cmd = (f'python -m experiments.run_multimodel_inference '
               f'--arms {arm} --models {model} --n {N_QUESTIONS}')
        r2_cmds.append((cmd, f'r2_{arm}_{model.replace(".","_")}'))
parallel_run(r2_cmds, max_workers=MAX_WORKERS_R2)

## 2.6 Monitor — check per-combo progress

Run trong cell riêng (background) để xem progress real-time.

In [ ]:
import json
from pathlib import Path

def progress():
    for root_name, root in [('R1', 'data/eval/results'), ('R2', 'data/eval/multimodel/results')]:
        rp = Path(root)
        if not rp.exists(): continue
        print(f'\n{root_name}:')
        for d in sorted(rp.iterdir()):
            if d.is_dir():
                n = len([f for f in d.glob('A*.json') if not f.name.endswith('.error.json')])
                err = len(list(d.glob('A*.error.json')))
                err_tag = f' ({err} errors)' if err else ''
                print(f'  {d.name:<40} {n}/{N_QUESTIONS}{err_tag}')

progress()

## 2.7 Check API error pattern

Records với prompt_tokens=0 + completion_tokens=0 = silent OpenAI connection error (pipeline bug — không retry). Academic metrics will ignore judge-based filtering.

In [ ]:
from collections import Counter
for root in ['data/eval/results', 'data/eval/multimodel/results']:
    for d in Path(root).iterdir() if Path(root).exists() else []:
        if not d.is_dir(): continue
        n_total = n_zero = 0
        for f in d.glob('A*.json'):
            if f.name.endswith('.error.json'): continue
            r = json.loads(f.read_text(encoding='utf-8'))
            n_total += 1
            if r.get('prompt_tokens') == 0 and r.get('completion_tokens') == 0:
                n_zero += 1
        if n_zero > 0:
            print(f'⚠ {d.name}: {n_zero}/{n_total} API errors (zero-token)')

## 2.8 Backup output → GDrive

Sync results sang GDrive (chậm hơn `/content` nhưng persistent qua session reboot).

In [ ]:
GDRIVE_BACKUP = '/content/drive/MyDrive/legal-graph-kb-results/data/eval'
!mkdir -p $GDRIVE_BACKUP
!rsync -av --update data/eval/ $GDRIVE_BACKUP/ | tail -20
!du -sh $GDRIVE_BACKUP

## Done — Move to `03_colab_metrics_report.ipynb`